## Cell 1: Setup & Environment

In [ ]:
!pip install -q python-dotenv tqdm tenacity aiohttp aiofiles pyarrow

import os
import json
import re
import asyncio
import aiohttp
import aiofiles
import pandas as pd
from datetime import datetime
from pathlib import Path
from tqdm.asyncio import tqdm as atqdm
from tqdm import tqdm
from tenacity import (
    retry, stop_after_attempt,
    wait_exponential, retry_if_exception_type
)

# ── API KEY ──────────────────────────────────────────────────────────────────
try:
    from kaggle_secrets import UserSecretsClient
    OPENROUTER_API_KEY = UserSecretsClient().get_secret("OPENROUTER_API_KEY")
    print("API Key loaded from Kaggle Secrets.")
except Exception:
    try:
        from dotenv import load_dotenv
        load_dotenv()
        OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
        print("API Key loaded from .env (local).")
    except Exception:
        OPENROUTER_API_KEY = None

if not OPENROUTER_API_KEY:
    raise ValueError(
        "OPENROUTER_API_KEY not found. "
        "Set it in Kaggle Add-ons → Secrets, or in a local .env file."
    )

print("Setup complete.")

## Cell 2: Config  (chinh tai day)

In [ ]:
# ── INPUT FILES ──────────────────────────────────────────────────────────────
SRC_FILE_PATH        = "/kaggle/input/datasets/quangminh2401/testing-llm-as-judge/QA_data1.json"        # English source
TRANSLATED_FILE_PATH = "/kaggle/input/datasets/quangminh2401/testing-llm-as-judge/vi_QA_data1.json"     # Vietnamese translation

# ── OUTPUT ───────────────────────────────────────────────────────────────────
_base_name   = Path(SRC_FILE_PATH).stem
OUTPUT_DIR   = f"/kaggle/working/{_base_name}_eval"
os.makedirs(OUTPUT_DIR, exist_ok=True)

CHECKPOINT_JSONL = os.path.join(OUTPUT_DIR, f"checkpoint_{_base_name}.jsonl")
FINAL_CSV        = os.path.join(OUTPUT_DIR, f"eval_{_base_name}.csv")
FINAL_HTML       = os.path.join(OUTPUT_DIR, f"report_{_base_name}.html")

# ── COLUMNS ──────────────────────────────────────────────────────────────────
# Các cột đưa vào judge (bỏ qua Index và các cột meta khác)
COLUMNS_TO_EVAL = ["QA_Type", "question", "answer", "Scoring_Points"]

# ── JUDGE MODELS (OpenRouter) ─────────────────────────────────────────────────
JUDGE_MODELS = [
    "anthropic/claude-opus-4-5",
    "openai/gpt-4.1",
    "google/gemini-2.5-pro-preview-03-25",
]

# ── ASYNC SETTINGS ────────────────────────────────────────────────────────────
CONCURRENT_REQUESTS  = 5    # semaphore per model
REQUEST_TIMEOUT      = 120  # seconds

# ── AGGREGATION ───────────────────────────────────────────────────────────────
# final_verdict = REVIEW nếu fail_count >= FAIL_THRESHOLD HOẶC có critical
FAIL_THRESHOLD = 2
MAX_TOKENS = 2048

print(f"Output dir : {OUTPUT_DIR}")
print(f"Judge models: {JUDGE_MODELS}")
print(f"Columns     : {COLUMNS_TO_EVAL}")

## Cell 3: Prompts  (chinh tai day)

In [ ]:
SYSTEM_PROMPT = """
You are a senior medical translation evaluator specializing in English-to-Vietnamese clinical content.

You will be given a SOURCE (English) and a TRANSLATION (Vietnamese) of a medical QA record.
The translation has already been refined once by an LLM — expect generally acceptable quality.
Your job is to catch what slipped through.

Evaluate across 4 criteria (score 0–10 each):
- accuracy:     Is the medical meaning preserved exactly? (negations, dosages, comparisons)
- terminology:  Are medical terms correctly translated or appropriately kept in English?
- fluency:      Is the Vietnamese natural and unambiguous?
- completeness: Is anything added, omitted, or distorted?

Flag critical_error = true if ANY of the following apply:
- Wrong drug name, dosage, or unit
- Negation flipped (e.g. "no fever" → "có sốt")
- Answer key changed
- Terminology error that could cause clinical misinterpretation

Short fields (single letters, abbreviations, numbers) — do not penalize.
Array fields — evaluate each element but report as one comment.

OUTPUT: valid JSON only, no markdown, no explanation outside the JSON.
{
  "verdict": "PASS" or "FAIL",
  "critical": true or false,
  "scores": {
    "accuracy": <0-10>,
    "terminology": <0-10>,
    "fluency": <0-10>,
    "completeness": <0-10>
  },
  "comment": "<2-3 sentences max, focus on errors only, skip praise>"
}
"""

USER_PROMPT_TEMPLATE = """
SOURCE (English):
{src_json}

TRANSLATION (Vietnamese):
{translated_json}
"""

print("Prompts ready.")

## Cell 4: Core Functions

In [ ]:
# =============================================================================
# 4A -- FILE LOADER
# =============================================================================

def _read_file(path: str) -> list[dict]:
    # Doc JSON / CSV / JSONL / Parquet -> list of dicts
    ext = Path(path).suffix.lower()
    if ext == ".json":
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)
        return data if isinstance(data, list) else [data]
    elif ext == ".csv":
        return pd.read_csv(path).to_dict(orient="records")
    elif ext == ".jsonl":
        with open(path, "r", encoding="utf-8") as f:
            return [json.loads(l) for l in f if l.strip()]
    elif ext in (".parquet", ".pq"):
        return pd.read_parquet(path).to_dict(orient="records")
    else:
        raise ValueError(f"Unsupported file format: {ext}")


def load_samples(
    src_path: str,
    translated_path: str,
    columns: list[str]
) -> list[dict]:
    # Load 2 file, zip theo row index
    # Returns list of {_sidx, src_json, translated_json}
    src_rows  = _read_file(src_path)
    tran_rows = _read_file(translated_path)

    if len(src_rows) != len(tran_rows):
        raise ValueError(
            f"Row count mismatch: src={len(src_rows)}, translated={len(tran_rows)}. "
            "Hai file phai co cung so hang."
        )

    samples = []
    for idx, (src, tran) in enumerate(zip(src_rows, tran_rows)):
        src_subset  = {k: src.get(k)  for k in columns if k in src}
        tran_subset = {k: tran.get(k) for k in columns if k in tran}
        samples.append({
            "_sidx"          : idx,
            "src_json"       : json.dumps(src_subset,  ensure_ascii=False),
            "translated_json": json.dumps(tran_subset, ensure_ascii=False),
        })

    print(f"Loaded {len(samples)} samples.")
    return samples


# =============================================================================
# 4B -- CHECKPOINT MANAGER
# =============================================================================

def load_checkpoint(jsonl_path: str) -> dict:
    # Doc checkpoint JSONL -> dict {_sidx: result}
    results = {}
    if not os.path.exists(jsonl_path):
        return results
    with open(jsonl_path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                try:
                    rec = json.loads(line)
                    results[rec["_sidx"]] = rec
                except Exception:
                    pass
    return results


async def write_checkpoint(jsonl_path: str, record: dict, lock: asyncio.Lock):
    # Ghi 1 ket qua vao JSONL (thread-safe)
    async with lock:
        async with aiofiles.open(jsonl_path, "a", encoding="utf-8") as f:
            await f.write(json.dumps(record, ensure_ascii=False) + "\n")


# =============================================================================
# 4C -- TOKEN & STATS TRACKER
# =============================================================================

class StatsTracker:
    def __init__(self, models: list[str]):
        self.lock = asyncio.Lock()
        self.token_stats = {
            m: {"calls": 0, "prompt": 0, "completion": 0, "parse_errors": 0}
            for m in models
        }
        self.verdict_counts = {"PASS": 0, "REVIEW": 0}
        self.critical_count = 0

    async def record_call(self, model: str, usage: dict):
        async with self.lock:
            s = self.token_stats[model]
            s["calls"]      += 1
            s["prompt"]     += usage.get("prompt_tokens", 0)
            s["completion"] += usage.get("completion_tokens", 0)

    async def record_parse_error(self, model: str):
        async with self.lock:
            self.token_stats[model]["parse_errors"] += 1

    async def record_verdict(self, final_verdict: str, has_critical: bool):
        async with self.lock:
            self.verdict_counts[final_verdict] = \
                self.verdict_counts.get(final_verdict, 0) + 1
            if has_critical:
                self.critical_count += 1


# =============================================================================
# 4D -- CUSTOM EXCEPTIONS
# =============================================================================

class InsufficientCreditsError(Exception):
    # Het credit OpenRouter -- dung toan bo pipeline
    pass

class RateLimitError(Exception):
    # Rate limited -- retry voi backoff dai hon
    pass


# =============================================================================
# 4E -- API CALLER
# =============================================================================

def _parse_judge_response(raw: str) -> dict:
    # Parse JSON tu model response, strip markdown fences neu co
    raw = raw.strip()
    raw = re.sub(r"^```(?:json)?\s*", "", raw)
    raw = re.sub(r"\s*```$", "", raw)
    return json.loads(raw)


def _validate_judge_result(r: dict) -> dict:
    # Normalize output cua judge: verdict / critical / scores / comment
    verdict = str(r.get("verdict", "FAIL")).strip().upper()
    r["verdict"] = "PASS" if verdict == "PASS" else "FAIL"

    raw_crit = r.get("critical", False)
    if isinstance(raw_crit, str):
        r["critical"] = raw_crit.strip().lower() == "true"
    else:
        r["critical"] = bool(raw_crit)

    scores = r.get("scores") or {}
    for k in ("accuracy", "terminology", "fluency", "completeness"):
        scores.setdefault(k, None)
    r["scores"] = scores

    r.setdefault("comment", "")
    return r


def _extract_content(data: dict) -> str:
    # Extract content tu OpenRouter response
    # Check API-level error truoc (402/429/404/400)
    # Handle reasoning model (Gemini 3.1+): content co the la None
    if "error" in data:
        code = data["error"].get("code")
        msg  = data["error"].get("message", "unknown error")
        if code == 402:
            raise InsufficientCreditsError(f"[402] {msg}")
        elif code == 429:
            raise RateLimitError(f"[429] {msg}")
        elif code in (400, 404):
            raise ValueError(f"[{code}] {msg}")
        else:
            raise ValueError(f"[API error {code}] {msg}")

    message = data["choices"][0]["message"]
    content = message.get("content")

    # Reasoning model (Gemini 3.1+): content co the la None
    # JSON thuc nam o reasoning_details
    if content is None:
        reasoning_details = message.get("reasoning_details") or []
        for block in reversed(reasoning_details):
            if block.get("type") == "text" and block.get("text"):
                content = block["text"]
                break

    if not content:
        raise ValueError(
            f"[null content] model returned no content. "
            f"message keys: {list(message.keys())}"
        )

    return content


async def call_judge(
    session    : aiohttp.ClientSession,
    model      : str,
    src_json   : str,
    tran_json  : str,
    semaphore  : asyncio.Semaphore,
    stats      : StatsTracker,
) -> dict:
    # Goi 1 judge model, tra ve parsed + validated result dict

    user_prompt = USER_PROMPT_TEMPLATE.format(
        src_json=src_json,
        translated_json=tran_json
    )
    payload = {
        "model"  : model,
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": user_prompt},
        ],
        "response_format": {"type": "json_object"},
        "max_tokens": MAX_TOKENS,
    }
    headers = {
        "Authorization": f"Bearer {OPENROUTER_API_KEY}",
        "Content-Type" : "application/json",
    }

    @retry(
        stop=stop_after_attempt(3),
        wait=wait_exponential(multiplier=2, min=5, max=60),
        retry=retry_if_exception_type(
            (aiohttp.ClientError, asyncio.TimeoutError, RateLimitError)
        ),
        reraise=True,
    )
    async def _post():
        async with semaphore:
            async with session.post(
                "https://openrouter.ai/api/v1/chat/completions",
                headers=headers,
                json=payload,
                timeout=aiohttp.ClientTimeout(total=REQUEST_TIMEOUT),
            ) as resp:
                return await resp.json()

    def _make_error_result(comment: str) -> dict:
        return {
            "model"   : model,
            "verdict" : "FAIL",
            "critical": False,
            "scores"  : {k: None for k in
                         ("accuracy", "terminology", "fluency", "completeness")},
            "comment" : comment,
        }

    try:
        data    = await _post()
        usage   = data.get("usage", {})
        await stats.record_call(model, usage)

        content = _extract_content(data)
        result  = _parse_judge_response(content)
        result  = _validate_judge_result(result)
        result["model"] = model
        return result

    except InsufficientCreditsError:
        raise  # propagate len main() de dung pipeline

    except RateLimitError as e:
        await stats.record_parse_error(model)
        return _make_error_result(f"[rate_limit] exhausted retries: {e}")

    except (json.JSONDecodeError, ValueError) as e:
        await stats.record_parse_error(model)
        return _make_error_result(f"[parse] {type(e).__name__}: {str(e)[:200]}")

    except Exception as e:
        await stats.record_parse_error(model)
        return _make_error_result(f"[exception] {type(e).__name__}: {str(e)[:200]}")


# =============================================================================
# 4F -- AGGREGATOR
# =============================================================================

def aggregate(judge_results: list[dict]) -> dict:
    # Tach judge hop le (khong co parse error) va judge loi
    # Parse error duoc nhan biet qua comment bat dau bang "["
    valid   = [r for r in judge_results if not r.get("comment", "").startswith("[")]
    errored = [r for r in judge_results if r.get("comment", "").startswith("[")]
    error_count = len(errored)

    # Khong du judge hop le de ket luan -> REVIEW + danh dau incomplete
    if len(valid) < 2:
        return {
            "final_verdict": "REVIEW",
            "has_critical" : False,
            "fail_count"   : 0,
            "incomplete"   : True,
            "error_count"  : error_count,
        }

    has_critical = any(r.get("critical", False) for r in valid)
    fail_count   = sum(1 for r in valid if r.get("verdict") == "FAIL")

    # Tinh threshold tren so judge hop le thay vi co dinh tong so judges
    # Vi du: 2 valid judges -> threshold = round(2 * 2/3) = 1
    threshold = max(1, round(len(valid) * FAIL_THRESHOLD / len(JUDGE_MODELS)))

    if fail_count >= threshold or has_critical:
        final_verdict = "REVIEW"
    else:
        final_verdict = "PASS"

    return {
        "final_verdict": final_verdict,
        "has_critical" : has_critical,
        "fail_count"   : fail_count,
        "incomplete"   : error_count > 0,
        "error_count"  : error_count,
    }


print("Core functions loaded.")


## Cell 5: Main Async Runner

In [ ]:
async def process_sample(
    sample     : dict,
    session    : aiohttp.ClientSession,
    semaphores : dict,
    stats      : StatsTracker,
    ckpt_lock  : asyncio.Lock,
) -> dict:
    # Chay 3 judges song song cho 1 sample, aggregate, ghi checkpoint
    sidx      = sample["_sidx"]
    src_json  = sample["src_json"]
    tran_json = sample["translated_json"]

    judge_results = await asyncio.gather(*[
        call_judge(
            session   = session,
            model     = model,
            src_json  = src_json,
            tran_json = tran_json,
            semaphore = semaphores[model],
            stats     = stats,
        )
        for model in JUDGE_MODELS
    ])

    agg = aggregate(judge_results)

    record = {
        "_sidx"        : sidx,
        "final_verdict": agg["final_verdict"],
        "has_critical" : agg["has_critical"],
        "fail_count"   : agg["fail_count"],
        "incomplete"   : agg["incomplete"],
        "error_count"  : agg["error_count"],
    }
    for i, result in enumerate(judge_results, start=1):
        record[f"judge{i}_eval"] = json.dumps(result, ensure_ascii=False)

    await write_checkpoint(CHECKPOINT_JSONL, record, ckpt_lock)
    await stats.record_verdict(agg["final_verdict"], agg["has_critical"])
    return record


async def main():
    all_samples = load_samples(SRC_FILE_PATH, TRANSLATED_FILE_PATH, COLUMNS_TO_EVAL)

    done    = load_checkpoint(CHECKPOINT_JSONL)
    pending = [s for s in all_samples if s["_sidx"] not in done]
    print(f"Total: {len(all_samples)} | Done: {len(done)} | Pending: {len(pending)}")

    if not pending:
        print("All samples processed. Skipping API calls.")
        return

    stats      = StatsTracker(JUDGE_MODELS)
    ckpt_lock  = asyncio.Lock()
    semaphores = {m: asyncio.Semaphore(CONCURRENT_REQUESTS) for m in JUDGE_MODELS}

    connector = aiohttp.TCPConnector(limit=CONCURRENT_REQUESTS * len(JUDGE_MODELS))
    async with aiohttp.ClientSession(connector=connector) as session:
        tasks = [
            process_sample(s, session, semaphores, stats, ckpt_lock)
            for s in pending
        ]
        for coro in atqdm(asyncio.as_completed(tasks),
                          total=len(tasks), desc="Evaluating"):
            await coro

    # -- System Summary --------------------------------------------------------
    print("\n" + "=" * 56)
    print("  SYSTEM SUMMARY")
    print("=" * 56)

    print("\nTOKEN USAGE:")
    for model, s in stats.token_stats.items():
        short = model.split("/")[-1]
        print(f"  [{short}]  calls={s['calls']}  "
              f"prompt={s['prompt']:,}  completion={s['completion']:,}  "
              f"parse_errors={s['parse_errors']}")

    all_done     = load_checkpoint(CHECKPOINT_JSONL)
    total_pass   = sum(1 for r in all_done.values() if r["final_verdict"] == "PASS")
    total_review = sum(1 for r in all_done.values() if r["final_verdict"] == "REVIEW")
    total_crit   = sum(1 for r in all_done.values() if r["has_critical"])
    total        = len(all_done)

    print("\nVERDICT STATS:")
    print(f"  Total  : {total}")
    if total:
        print(f"  PASS   : {total_pass}  ({total_pass/total*100:.1f}%)")
        print(f"  REVIEW : {total_review}  ({total_review/total*100:.1f}%)")
    print(f"  Critical flags: {total_crit}")
    print("=" * 56)


try:
    await main()
except InsufficientCreditsError as e:
    print(f"\nPIPELINE STOPPED: {e}")
    print("Checkpoint saved. Resume after adding credits.")
except KeyboardInterrupt:
    print("\nInterrupted by user. Checkpoint saved.")


## Cell 6: Export CSV + HTML Report

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# 6A — LOAD CHECKPOINT → SORTED DATAFRAME
# ═══════════════════════════════════════════════════════════════════════════════

def build_dataframe() -> pd.DataFrame:
    done = load_checkpoint(CHECKPOINT_JSONL)
    if not done:
        raise ValueError("No data in checkpoint. Run Cell 5 first.")
    df = pd.DataFrame(list(done.values()))
    df = df.sort_values("_sidx").reset_index(drop=True)
    return df


# ═══════════════════════════════════════════════════════════════════════════════
# 6B — EXPORT CSV
# ═══════════════════════════════════════════════════════════════════════════════

def export_csv(df: pd.DataFrame):
    cols = ["_sidx", "final_verdict", "has_critical", "fail_count",
            "incomplete", "error_count"] + \
           [f"judge{i}_eval" for i in range(1, len(JUDGE_MODELS) + 1)]
    df[cols].to_csv(FINAL_CSV, index=False, encoding="utf-8-sig")
    print(f"CSV saved: {FINAL_CSV}")


# ═══════════════════════════════════════════════════════════════════════════════
# 6C — EXPORT HTML REPORT
# ═══════════════════════════════════════════════════════════════════════════════

def _render_judge_cell(eval_json_str: str, judge_idx: int) -> str:
    """Render 1 ô judge thành HTML có thể expand."""
    try:
        r = json.loads(eval_json_str)
    except Exception:
        return f'<span class="parse-err">parse error</span>'

    verdict  = r.get("verdict", "?")
    critical = r.get("critical", False)
    scores   = r.get("scores", {})
    comment  = r.get("comment", "")
    model    = r.get("model", f"judge{judge_idx}").split("/")[-1]

    v_class  = "badge-pass" if verdict == "PASS" else "badge-fail"
    crit_tag = ' <span class="crit-icon">[!]</span>' if critical else ""

    scores_html = "".join(
        f'<div class="score-row"><span class="score-label">{k}</span>'
        f'<span class="score-bar"><span style="width:{(v or 0)*10}%" '
        f'class="score-fill"></span></span>'
        f'<span class="score-val">{v if v is not None else "—"}</span></div>'
        for k, v in scores.items()
    )
    uid = f"j{judge_idx}_{id(eval_json_str) & 0xFFFF}"
    return f"""
    <div class="judge-card">
      <div class="judge-header">
        <span class="model-name">{model}</span>
        <span class="badge {v_class}">{verdict}</span>{crit_tag}
        <button class="toggle-btn" onclick="toggleDetail('{uid}')">▾</button>
      </div>
      <div class="judge-detail" id="{uid}" style="display:none">
        {scores_html}
        <div class="comment">{comment}</div>
      </div>
    </div>"""


def export_html(df: pd.DataFrame):
    total        = len(df)
    total_pass   = (df["final_verdict"] == "PASS").sum()
    total_review = (df["final_verdict"] == "REVIEW").sum()
    total_crit   = df["has_critical"].sum()
    run_date     = datetime.now().strftime("%Y-%m-%d %H:%M")
    src_name     = Path(SRC_FILE_PATH).name

    pass_pct   = total_pass   / total * 100 if total else 0
    review_pct = total_review / total * 100 if total else 0

    # ── Rows ──────────────────────────────────────────────────────────────────
    rows_html = ""
    for _, row in df.iterrows():
        fv       = row["final_verdict"]
        crit     = row["has_critical"]
        row_cls  = "row-review" if fv == "REVIEW" else ""
        crit_cls = " row-critical" if crit else ""
        incomplete  = bool(row.get("incomplete", False))
        badge       = f'<span class="badge badge-{fv.lower()}">{fv}</span>'
        crit_badge  = ' <span class="crit-icon">[!]</span>' if crit else ""
        incomp_badge = ' <span class="incomp-badge">partial</span>' if incomplete else ""

        judges_html = "".join(
            _render_judge_cell(row.get(f"judge{i}_eval", "{}"), i)
            for i in range(1, len(JUDGE_MODELS) + 1)
        )
        rows_html += f"""
        <tr class="{row_cls}{crit_cls}" data-verdict="{fv}" data-critical="{'1' if crit else '0'}">
          <td class="idx-cell">{int(row['_sidx'])}</td>
          <td>{badge}{crit_badge}{incomp_badge}</td>
          <td class="judges-cell">{judges_html}</td>
        </tr>"""

    # ── Full HTML ─────────────────────────────────────────────────────────────
    html = f"""<!DOCTYPE html>
<html lang="vi">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<title>Translation QA Report — {src_name}</title>
<style>
  @import url('https://fonts.googleapis.com/css2?family=JetBrains+Mono:wght@400;600&family=Sora:wght@300;400;600;700&display=swap');

  :root {{
    --bg         : #0f1117;
    --surface    : #1a1d27;
    --surface2   : #22263a;
    --border     : #2e3150;
    --pass       : #22c55e;
    --pass-dim   : #14532d22;
    --review     : #f59e0b;
    --review-dim : #78350f22;
    --critical   : #ef4444;
    --crit-dim   : #7f1d1d18;
    --text       : #e2e8f0;
    --muted      : #64748b;
    --accent     : #6366f1;
  }}

  * {{ box-sizing: border-box; margin: 0; padding: 0; }}

  body {{
    font-family: 'Sora', sans-serif;
    background: var(--bg);
    color: var(--text);
    min-height: 100vh;
    padding: 2rem;
  }}

  /* ── Header ── */
  .header {{
    display: flex; align-items: flex-start;
    justify-content: space-between; flex-wrap: wrap;
    gap: 1rem; margin-bottom: 2.5rem;
    border-bottom: 1px solid var(--border);
    padding-bottom: 1.5rem;
  }}
  .header-title {{
    font-size: 1.5rem; font-weight: 700; letter-spacing: -0.02em;
  }}
  .header-title span {{ color: var(--accent); }}
  .header-meta {{ font-size: 0.78rem; color: var(--muted); margin-top: 0.3rem; }}
  .header-meta b {{ color: var(--text); }}

  /* ── Summary cards ── */
  .summary {{
    display: grid;
    grid-template-columns: repeat(auto-fit, minmax(160px, 1fr));
    gap: 1rem; margin-bottom: 2rem;
  }}
  .stat-card {{
    background: var(--surface);
    border: 1px solid var(--border);
    border-radius: 12px; padding: 1.2rem 1.4rem;
  }}
  .stat-card .stat-val {{
    font-family: 'JetBrains Mono', monospace;
    font-size: 2rem; font-weight: 600;
  }}
  .stat-card .stat-label {{ font-size: 0.75rem; color: var(--muted); margin-top: 0.25rem; }}
  .stat-card.c-pass  .stat-val {{ color: var(--pass); }}
  .stat-card.c-review .stat-val {{ color: var(--review); }}
  .stat-card.c-crit  .stat-val {{ color: var(--critical); }}

  /* progress bar summary */
  .progress-bar {{
    height: 6px; border-radius: 3px;
    background: var(--surface2);
    margin-top: 0.6rem; overflow: hidden;
  }}
  .progress-fill-pass   {{ height: 100%; background: var(--pass);   border-radius: 3px; }}
  .progress-fill-review {{ height: 100%; background: var(--review); border-radius: 3px; }}

  /* ── Filters ── */
  .filters {{
    display: flex; gap: 0.5rem; margin-bottom: 1.5rem; flex-wrap: wrap;
  }}
  .filter-btn {{
    font-family: 'Sora', sans-serif;
    font-size: 0.78rem; font-weight: 600;
    padding: 0.4rem 1rem; border-radius: 999px;
    border: 1px solid var(--border);
    background: var(--surface); color: var(--muted);
    cursor: pointer; transition: all .15s;
  }}
  .filter-btn:hover, .filter-btn.active {{
    background: var(--accent); color: #fff; border-color: var(--accent);
  }}

  /* ── Table ── */
  .table-wrap {{
    overflow-x: auto;
    border: 1px solid var(--border);
    border-radius: 12px;
  }}
  table {{
    width: 100%; border-collapse: collapse;
    font-size: 0.85rem;
  }}
  thead th {{
    background: var(--surface2);
    padding: 0.75rem 1rem;
    text-align: left; font-weight: 600;
    font-size: 0.75rem; text-transform: uppercase;
    letter-spacing: 0.06em; color: var(--muted);
    border-bottom: 1px solid var(--border);
  }}
  tbody tr {{
    border-bottom: 1px solid var(--border);
    transition: background .1s;
  }}
  tbody tr:hover {{ background: var(--surface2); }}
  tbody tr.row-review {{ background: var(--review-dim); }}
  tbody tr.row-critical {{ background: var(--crit-dim) !important; }}
  td {{ padding: 0.7rem 1rem; vertical-align: top; }}
  .idx-cell {{
    font-family: 'JetBrains Mono', monospace;
    font-size: 0.78rem; color: var(--muted);
    width: 56px;
  }}

  /* ── Badge ── */
  .badge {{
    display: inline-block;
    font-family: 'JetBrains Mono', monospace;
    font-size: 0.7rem; font-weight: 600;
    padding: 0.2rem 0.6rem; border-radius: 5px;
    letter-spacing: 0.05em;
  }}
  .badge-pass   {{ background: var(--pass-dim);   color: var(--pass);    border: 1px solid #22c55e44; }}
  .badge-review {{ background: var(--review-dim); color: var(--review);  border: 1px solid #f59e0b44; }}
  .badge-fail   {{ background: var(--review-dim); color: var(--review);  border: 1px solid #f59e0b44; }}

  .crit-icon {{ font-size: 0.85rem; margin-left: 4px; }}
  .incomp-badge {{
    display: inline-block;
    font-family: 'JetBrains Mono', monospace;
    font-size: 0.65rem; font-weight: 600;
    padding: 0.15rem 0.5rem; border-radius: 4px;
    margin-left: 5px;
    background: #1e293b; color: #94a3b8;
    border: 1px solid #334155;
    letter-spacing: 0.04em;
  }}

  /* ── Judge cards ── */
  .judges-cell {{ min-width: 320px; }}
  .judge-card {{
    background: var(--surface);
    border: 1px solid var(--border);
    border-radius: 8px; margin-bottom: 0.5rem;
    overflow: hidden;
  }}
  .judge-header {{
    display: flex; align-items: center; gap: 0.5rem;
    padding: 0.5rem 0.75rem;
  }}
  .model-name {{
    font-family: 'JetBrains Mono', monospace;
    font-size: 0.72rem; color: var(--muted);
    flex: 1; white-space: nowrap; overflow: hidden;
    text-overflow: ellipsis;
  }}
  .toggle-btn {{
    background: none; border: none; color: var(--muted);
    cursor: pointer; font-size: 0.85rem; padding: 0 4px;
    transition: transform .2s;
  }}
  .judge-detail {{
    padding: 0.6rem 0.75rem;
    border-top: 1px solid var(--border);
  }}

  /* ── Score bars ── */
  .score-row {{
    display: flex; align-items: center;
    gap: 0.5rem; margin-bottom: 0.35rem;
  }}
  .score-label {{
    font-size: 0.68rem; color: var(--muted);
    width: 88px; flex-shrink: 0;
  }}
  .score-bar {{
    flex: 1; height: 5px; background: var(--surface2);
    border-radius: 3px; overflow: hidden;
  }}
  .score-fill {{
    display: block; height: 100%;
    background: var(--accent); border-radius: 3px;
    transition: width .4s ease;
  }}
  .score-val {{
    font-family: 'JetBrains Mono', monospace;
    font-size: 0.7rem; width: 20px; text-align: right;
  }}
  .comment {{
    font-size: 0.78rem; color: #94a3b8;
    margin-top: 0.5rem; line-height: 1.55;
    border-top: 1px solid var(--border);
    padding-top: 0.45rem;
  }}
  .parse-err {{ color: var(--critical); font-size: 0.75rem; }}

  /* ── Footer ── */
  .footer {{
    margin-top: 3rem; text-align: center;
    font-size: 0.72rem; color: var(--muted);
  }}
</style>
</head>
<body>

<div class="header">
  <div>
    <div class="header-title">Medical Translation <span>QA Report</span></div>
    <div class="header-meta">
      <b>Source:</b> {src_name} &nbsp;·&nbsp;
      <b>Models:</b> {', '.join(m.split('/')[-1] for m in JUDGE_MODELS)} &nbsp;·&nbsp;
      <b>Run:</b> {run_date}
    </div>
  </div>
</div>

<div class="summary">
  <div class="stat-card">
    <div class="stat-val">{total}</div>
    <div class="stat-label">Total Samples</div>
  </div>
  <div class="stat-card c-pass">
    <div class="stat-val">{total_pass}</div>
    <div class="stat-label">PASS &nbsp;({pass_pct:.1f}%)</div>
    <div class="progress-bar"><div class="progress-fill-pass" style="width:{pass_pct:.1f}%"></div></div>
  </div>
  <div class="stat-card c-review">
    <div class="stat-val">{total_review}</div>
    <div class="stat-label">REVIEW &nbsp;({review_pct:.1f}%)</div>
    <div class="progress-bar"><div class="progress-fill-review" style="width:{review_pct:.1f}%"></div></div>
  </div>
  <div class="stat-card c-crit">
    <div class="stat-val">{total_crit}</div>
    <div class="stat-label">Critical Flags</div>
  </div>
</div>

<div class="filters">
  <button class="filter-btn active" onclick="filterRows('ALL', this)">All ({total})</button>
  <button class="filter-btn" onclick="filterRows('REVIEW', this)">Review ({total_review})</button>
  <button class="filter-btn" onclick="filterRows('CRITICAL', this)">Critical ({total_crit})</button>
  <button class="filter-btn" onclick="filterRows('PASS', this)">Pass ({total_pass})</button>
</div>

<div class="table-wrap">
  <table>
    <thead>
      <tr>
        <th>#</th>
        <th>Verdict</th>
        <th>Judge Evaluations</th>
      </tr>
    </thead>
    <tbody id="tbl-body">
      {rows_html}
    </tbody>
  </table>
</div>

<div class="footer">
  Generated by LLM-as-a-Judge Pipeline &nbsp;·&nbsp; {run_date}
</div>

<script>
  function toggleDetail(uid) {{
    const el = document.getElementById(uid);
    if (!el) return;
    const open = el.style.display !== 'none';
    el.style.display = open ? 'none' : 'block';
    const btn = el.previousElementSibling?.querySelector('.toggle-btn');
    if (btn) btn.textContent = open ? '▾' : '▴';
  }}

  function filterRows(type, btn) {{
    document.querySelectorAll('.filter-btn').forEach(b => b.classList.remove('active'));
    btn.classList.add('active');
    document.querySelectorAll('#tbl-body tr').forEach(row => {{
      const verdict  = row.dataset.verdict;
      const critical = row.dataset.critical === '1';
      let show = false;
      if (type === 'ALL')      show = true;
      else if (type === 'PASS')     show = verdict === 'PASS';
      else if (type === 'REVIEW')   show = verdict === 'REVIEW';
      else if (type === 'CRITICAL') show = critical;
      row.style.display = show ? '' : 'none';
    }});
  }}
</script>
</body></html>"""

    with open(FINAL_HTML, "w", encoding="utf-8") as f:
        f.write(html)
    print(f"HTML report saved: {FINAL_HTML}")


# ── Chạy export ────────────────────────────────────────────────────────────────
df = build_dataframe()
export_csv(df)
export_html(df)

print("\nOutput files:")
print(f"  CSV  -> {FINAL_CSV}")
print(f"  HTML -> {FINAL_HTML}")

## Cell 7: Debug -- Raw HTTP Response

In [ ]:
# =============================================================================
# DEBUG -- Goi lai API cho 1 row, in toan bo raw HTTP response cua 3 judges
# =============================================================================
# Chinh bien nay de chon row can debug (theo thu tu trong file, bat dau tu 0)
DEBUG_ROW_INDEX = 0

# =============================================================================

async def debug_single_row(sidx: int):
    # Load samples de lay dung row
    all_samples = load_samples(SRC_FILE_PATH, TRANSLATED_FILE_PATH, COLUMNS_TO_EVAL)

    if sidx < 0 or sidx >= len(all_samples):
        print(f"Row index {sidx} khong hop le. File co {len(all_samples)} rows (0 den {len(all_samples)-1}).")
        return

    sample    = all_samples[sidx]
    src_json  = sample["src_json"]
    tran_json = sample["translated_json"]

    print("=" * 64)
    print(f"DEBUG ROW: _sidx = {sidx}")
    print("=" * 64)
    print("\nSOURCE JSON gui vao:")
    print(json.dumps(json.loads(src_json), ensure_ascii=False, indent=2))
    print("\nTRANSLATED JSON gui vao:")
    print(json.dumps(json.loads(tran_json), ensure_ascii=False, indent=2))

    user_prompt = USER_PROMPT_TEMPLATE.format(
        src_json=src_json,
        translated_json=tran_json
    )
    headers = {
        "Authorization": f"Bearer {OPENROUTER_API_KEY}",
        "Content-Type" : "application/json",
    }

    async def call_raw(model: str) -> dict:
        payload = {
            "model"  : model,
            "messages": [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user",   "content": user_prompt},
            ],
            "response_format": {"type": "json_object"},
            "max_tokens": MAX_TOKENS,
        }
        try:
            async with aiohttp.ClientSession() as session:
                async with session.post(
                    "https://openrouter.ai/api/v1/chat/completions",
                    headers=headers,
                    json=payload,
                    timeout=aiohttp.ClientTimeout(total=REQUEST_TIMEOUT),
                ) as resp:
                    status = resp.status
                    data   = await resp.json()
                    return {"http_status": status, "body": data}
        except Exception as e:
            return {"http_status": None, "body": None, "exception": f"{type(e).__name__}: {e}"}

    # Goi 3 judges song song
    raw_results = await asyncio.gather(*[call_raw(m) for m in JUDGE_MODELS])

    # In raw response tung judge
    for i, (model, raw) in enumerate(zip(JUDGE_MODELS, raw_results), start=1):
        print("\n" + "-" * 64)
        print(f"JUDGE {i}: {model}")
        print(f"HTTP status: {raw.get('http_status')}")
        print("-" * 64)

        if "exception" in raw:
            print(f"[EXCEPTION] {raw['exception']}")
            continue

        body = raw["body"]

        # In error block neu co
        if "error" in body:
            print("[API ERROR]")
            print(json.dumps(body["error"], ensure_ascii=False, indent=2))
            continue

        # In usage
        usage = body.get("usage", {})
        print(f"Usage: prompt={usage.get('prompt_tokens')}  "
              f"completion={usage.get('completion_tokens')}  "
              f"total={usage.get('total_tokens')}")

        # In message block day du
        try:
            message = body["choices"][0]["message"]
            print(f"finish_reason : {body['choices'][0].get('finish_reason')}")
            print(f"message.keys(): {list(message.keys())}")

            content = message.get("content")
            print(f"\ncontent (raw):")
            print(content if content is not None else "[None]")

            # In reasoning_details neu co
            rd = message.get("reasoning_details")
            if rd:
                print(f"\nreasoning_details ({len(rd)} blocks):")
                for j, block in enumerate(rd):
                    print(f"  block[{j}]: type={block.get('type')}  "
                          f"len={len(block.get('text','') or '')} chars")
                    if block.get("type") == "text":
                        preview = (block.get("text") or "")[:300]
                        print(f"  preview: {preview}")

        except (KeyError, IndexError) as e:
            print(f"[ERROR reading choices] {e}")
            print("Full body:")
            print(json.dumps(body, ensure_ascii=False, indent=2))

        # Thu parse de xem pipeline xu ly duoc khong
        print("\n-- Pipeline parse attempt --")
        try:
            content_to_parse = message.get("content")
            if content_to_parse is None:
                rd = message.get("reasoning_details") or []
                for block in reversed(rd):
                    if block.get("type") == "text" and block.get("text"):
                        content_to_parse = block["text"]
                        print("[used reasoning_details fallback]")
                        break
            if not content_to_parse:
                print("[FAIL] content rong sau fallback")
            else:
                parsed = _parse_judge_response(content_to_parse)
                validated = _validate_judge_result(parsed)
                print("[OK] Parse thanh cong:")
                print(json.dumps(validated, ensure_ascii=False, indent=2))
        except Exception as e:
            print(f"[FAIL] {type(e).__name__}: {e}")

    print("\n" + "=" * 64)
    print("DEBUG COMPLETE")
    print("=" * 64)


#await debug_single_row(DEBUG_ROW_INDEX)
